# Chapter 06 — Pandas I: Series, DataFrames, and Inspection

Pandas makes labelled, heterogeneous tabular data easier to inspect and manipulate. Focus on the data model, selection, indexing, dtypes, and descriptive summaries.

## Prerequisites
Python and NumPy basics; install Pandas.

## Learning goals
- Explain Series, DataFrame, index, columns, and alignment.
- Read tabular data and inspect its structure.
- Select rows and columns with loc and iloc.
- Understand dtypes, missing values, and descriptive summaries.

**Study sequence:** use the theory notebook first, then tutorial, then lab. The examples use the bundled synthetic data unless otherwise noted.

In [1]:
from pathlib import Path
ROOT=Path.cwd()
if not (ROOT/"data").exists() and (ROOT.parent/"data").exists(): ROOT=ROOT.parent
DATA=ROOT/"data"
print("Data directory:", DATA.resolve())

Data directory: /mnt/data/python_data_analysis_book/data


## Chapter map

1. Series and DataFrame
2. Reading tabular data
3. Selection and indexing
4. Data types and index design
5. First-pass inspection

### Series and DataFrame
A `Series` is a one-dimensional labelled array with an index. A `DataFrame` is a two-dimensional labelled table whose columns may have different dtypes. Useful attributes: `.index`, `.columns`, `.dtypes`, `.shape`, `.ndim`, `.size`, `.empty`. Attributes do not use parentheses; methods such as `.head()` do. Keep column labels unique.

#### Demonstration 1 — Create Series and DataFrame

Run the code and inspect both the output and the assumptions behind it.

In [2]:
import pandas as pd
s=pd.Series([80,90,75],index=["Asha","Kabir","Diya"],name="math")
df=pd.DataFrame({"name":["Asha","Kabir","Diya"],"math":[80,90,75],"city":["Kolkata","Delhi","Pune"]})
print(s); print(df.shape,df.columns.tolist(),df.dtypes.to_dict())

Asha     80
Kabir    90
Diya     75
Name: math, dtype: int64
(3, 3) ['name', 'math', 'city'] {'name': dtype('O'), 'math': dtype('int64'), 'city': dtype('O')}


### Reading tabular data
`pd.read_csv(filepath_or_buffer, sep=",", header="infer", names=None, index_col=None, usecols=None, dtype=None, parse_dates=None, na_values=None, encoding=None)` reads delimited text. `usecols` reduces columns, `dtype` controls type inference, `parse_dates` converts date fields, `na_values` adds missing markers. Other readers include `read_json`, `read_excel`, `read_parquet`, and `read_sql`.

#### Demonstration 2 — Load CSV with explicit date and identifier type

Run the code and inspect both the output and the assumptions behind it.

In [3]:
from pathlib import Path
ROOT=Path.cwd()
if not (ROOT/"data").exists() and (ROOT.parent/"data").exists(): ROOT=ROOT.parent
DATA=ROOT/"data"
print("Data directory:", DATA.resolve())
import pandas as pd
sales=pd.read_csv(DATA/"sales.csv",parse_dates=["order_date"],dtype={"order_id":"string"})
display(sales.head()); print(sales.dtypes)

Data directory: /mnt/data/python_data_analysis_book/data
  order_id order_date region  ...       cost    profit  customer_id
0    O0101 2025-01-02  North  ...  139372.86  80562.93         C011
1    O0143 2025-01-03   West  ...    7802.99   4092.40         C008
2    O0120 2025-01-04   West  ...   13006.05   3653.15         C015
3    O0188 2025-01-05  South  ...   13058.82   2575.22         C008
4    O0047 2025-01-07   East  ...    7562.05   1371.98         C002

[5 rows x 11 columns]
order_id       string[python]
order_date     datetime64[ns]
region                 object
product                object
units                   int64
unit_price            float64
discount              float64
revenue               float64
cost                  float64
profit                float64
customer_id            object
dtype: object


### Selection and indexing
`df["column"]` selects a Series; `df[["a","b"]]` selects multiple columns. `.loc[row_label, column_label]` is label-based; `.iloc[row_position, column_position]` is integer-position-based. `.loc` label slices include endpoint labels; `.iloc` follows Python-exclusive stop behavior. Use `.loc[mask, columns]` for explicit conditional selection and assignment.

#### Demonstration 3 — Select with loc and iloc

Run the code and inspect both the output and the assumptions behind it.

In [4]:
print(sales.loc[0,["order_id","region","revenue"]])
display(sales.iloc[:3,:4])
large=sales.loc[sales.revenue>10000,["order_id","region","revenue"]]; display(large.head())

order_id        O0101
region          North
revenue     219935.79
Name: 0, dtype: object
  order_id order_date region   product
0    O0101 2025-01-02  North   Monitor
1    O0143 2025-01-03   West     Mouse
2    O0120 2025-01-04   West  Keyboard
  order_id region     revenue
0    O0101  North   219935.79
1    O0143   West    11895.39
2    O0120   West    16659.20
3    O0188  South    15634.04
7    O0213   West  1053567.91


### Data types and index design
Common dtypes: `int64`, `float64`, `bool`, `object`, `string`, `category`, and `datetime64`. Nullable extension dtypes (`Int64`, `boolean`, `string`) can represent missing values. An index labels rows but is not automatically a primary key. `set_index` and `reset_index` move values between columns and index; `astype` converts validated types; `to_datetime(errors="coerce")` turns invalid dates into `NaT`.

#### Demonstration 4 — Inspect distributions and categories

Run the code and inspect both the output and the assumptions behind it.

In [5]:
sales.info()
display(sales[["units","revenue","profit"]].describe().T)
print(sales.region.value_counts())
print("missing:",sales.isna().sum().to_dict())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 240 entries, 0 to 239
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   order_id     240 non-null    string        
 1   order_date   240 non-null    datetime64[ns]
 2   region       240 non-null    object        
 3   product      240 non-null    object        
 4   units        240 non-null    int64         
 5   unit_price   240 non-null    float64       
 6   discount     240 non-null    float64       
 7   revenue      240 non-null    float64       
 8   cost         240 non-null    float64       
 9   profit       240 non-null    float64       
 10  customer_id  240 non-null    object        
dtypes: datetime64[ns](1), float64(5), int64(1), object(3), string(1)
memory usage: 20.8+ KB
         count           mean  ...          75%         max
units    240.0       9.591667  ...      13.0000       17.00
revenue  240.0  164247.956917  ...  164276.13

### First-pass inspection
`head(n=5)`, `tail(n=5)`, `info()`, `describe(include=..., exclude=...)`, `value_counts(dropna=True)`, `nunique(dropna=True)`, `isna`, and `notna` are core inspection methods. Numeric summaries do not prove data quality; separately validate ranges, allowed categories, uniqueness, dates, and foreign keys.

#### Demonstration 5 — Convert types and add a column

Run the code and inspect both the output and the assumptions behind it.

In [6]:
sales["region"]=sales["region"].astype("category")
sales=sales.assign(profit_margin=sales.profit/sales.revenue)
print(sales.dtypes); display(sales[["order_id","profit_margin"]].head())

order_id         string[python]
order_date       datetime64[ns]
region                 category
product                  object
units                     int64
unit_price              float64
discount                float64
revenue                 float64
cost                    float64
profit                  float64
customer_id              object
profit_margin           float64
dtype: object
  order_id  profit_margin
0    O0101       0.366302
1    O0143       0.344032
2    O0120       0.219287
3    O0188       0.164719
4    O0047       0.153568


## Self-check

Without looking back, explain the key terms, identify the important parameters, describe at least one alternative or variation, and state one limitation of the methods covered. Then answer the exercises below.

### Review questions

- Explain one important concept from: Series and DataFrame.
- Explain one important concept from: Reading tabular data.
- Explain one important concept from: Selection and indexing.
- Explain one important concept from: Data types and index design.
- Explain one important concept from: First-pass inspection.